# `temp_forecasting_pipeline.ipynb` -- forecasting testing/experimentation base

Condensed, tidied working base for the recursive-rollout forecasting phase (B-09-B16), mirroring
`notebooks/03c_gap_filling_revisited/temp_gap_filling_pipeline.ipynb`'s role for gap-filling: load
data once, define the shared rollout harness **inline** (fully self-contained -- zero `src/`
imports, every function replicated directly in this notebook, matching the gap-filling notebook's
own standing convention exactly), reproduce the standing champion live as a working smoke test,
consolidate every model/config this project has evaluated into one climatology-scored master
comparison table, then leave a scaffold section for new experiments going forward (the same role
D1-D8 played in the gap-filling notebook).

**Self-contained, hands-on**: every function this notebook uses (`chain_persistence`,
`doy_climatology`, `lead_time_bin`, `bin_metrics`, the metric primitives it depends on, plus
`tabpfn_forecast`/`tabicl_forecast`/`tabpfn_v2_model_config`) is defined in Section 3 below, not
imported from `src/models/recursive_rollout.py` or `src/evaluation/metrics.py`. Those `src/`
modules remain the production single-source-of-truth (used by the committed `b16_*.py` scripts);
this notebook is a separate, standalone re-derivation for fast, hands-on iteration -- the same
relationship the gap-filling notebook had to `src/models/gapfill_rfm.py` (notebook came first,
`src/` was extracted from it afterward, not the other way round).

**MASE convention: climatology, not persistence (D-80).** Everything in this notebook is scored
against `doy_climatology()`, not `chain_persistence()` -- see CLAUDE.md.

**Compute discipline**: TabPFN/TabICLv2 (zero-shot foundation models) are cheap (~150s for a full
3-tower x 5-anchor x 7-config sweep) and are rerun live here for genuine verification. Tree/
SARIMAX models and the DL family (TFT/DLinear/LSTM, which need real gradient-based training per
anchor/tower) are expensive and are **imported** from their already-computed, already-validated
result CSVs rather than refit -- clearly labelled at each import point (their own source scripts,
`notebooks/05_benchmarking/b16_recursive_rollout_v3_all.py`/`b16_dl_models_v3.py`, are themselves
`src/`-importing production scripts, out of scope for this notebook's self-containment goal, which
applies to what this notebook itself defines and runs live).

## 1. Setup

In [1]:
import sys
import time
import warnings
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from scipy.stats import linregress
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

ROOT = Path(r"C:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project")
load_dotenv(ROOT / ".env")

HOURLY = ROOT / "data" / "Hourly"
RESULTS = ROOT / "results"

TOWERS = [2, 4, 9]
N_DAYS = 365
ANCHOR_YEARS = [2018, 2019, 2020, 2021, 2022]
BINS = ((1, 7), (8, 30), (31, 90), (91, 180), (181, 270), (271, 365))

TABPFN_OK = bool(os.environ.get("TABPFN_TOKEN"))
print(f"TABPFN_TOKEN set: {TABPFN_OK}")

TABPFN_TOKEN set: True


## 2. Data

`forecast_daily_v3.csv` -- the F-10-enriched daily table (guide `fx_` features + daily AR
`ar_ch4_*`/`ar_fc_dlag1` + `y_observed`/`y_gapfilled` target, RFm-champion-sourced gap-fill).
Built by `src/features/build_forecasting_matrix_v2.py` + `v3.py` (not reproduced here -- read-only
consumer of a saved CSV, not a `src/` code import, so this doesn't affect self-containment).

In [2]:
dv = pd.read_csv(HOURLY / "forecast_daily_v3.csv", low_memory=False)
dv["Datetime"] = pd.to_datetime(dv["Datetime"], format="mixed")
fx_all = [c for c in dv.columns if c.startswith("fx")]
T = {t: dv[dv.tower == t].set_index("Datetime").sort_index() for t in TOWERS}
print(f"Loaded forecast_daily_v3.csv {dv.shape}, {len(fx_all)} fx_ columns, towers {TOWERS}")

Loaded forecast_daily_v3.csv (8772, 66), 52 fx_ columns, towers [2, 4, 9]


## 3. Shared rollout harness -- inlined, self-contained (no `src/` imports)

Every function below is a direct, faithful re-derivation of `src/models/recursive_rollout.py` +
`src/evaluation/metrics.py` (read directly from those files, not reimplemented from memory) --
this notebook does not `import` either module. Kept byte-logic-identical to the production
versions so results here match the committed `b16_*.py` scripts exactly; only the "reuse via
import" mechanism changes, not the methodology itself.

### 3.1 Metric primitives (`src/evaluation/metrics.py`)

In [3]:
def mae(y, p):
    return float(mean_absolute_error(y, p))


def rmse(y, p):
    return float(np.sqrt(mean_squared_error(y, p)))


def wape(y, p):
    """Weighted Absolute Percentage Error = sum|y-p| / sum|y|. Aggregates before dividing, so
    unlike MAPE a few near-zero actuals can't blow it up. NaN if sum|y| == 0."""
    y = np.asarray(y, float); p = np.asarray(p, float)
    denom = np.sum(np.abs(y))
    return float(np.sum(np.abs(y - p)) / denom) if denom > 0 else np.nan


def correlation(y, p):
    """Pearson r -- scale/bias-invariant, distinguishes "wrong scale, right pattern" from "no
    real signal", a distinction R2 alone conflates."""
    y = np.asarray(y, float); p = np.asarray(p, float)
    if len(y) < 2 or np.std(y) == 0 or np.std(p) == 0:
        return np.nan
    return float(np.corrcoef(y, p)[0, 1])


def mase(y, p, y_naive):
    """MASE, test-set relative-MAE form: MAE(model) / MAE(naive baseline). <1 = beats the
    baseline, 1 = ties it, >1 = worse. NaN if the baseline's own MAE is 0."""
    denom = mae(y, y_naive)
    return float(mae(y, p) / denom) if denom > 0 else np.nan


def rmsse(y, p, y_naive):
    """RMSSE: the squared-error analogue of mase() -- RMSE(model) / RMSE(naive baseline)."""
    denom = rmse(y, y_naive)
    return float(rmse(y, p) / denom) if denom > 0 else np.nan


print("Metric primitives defined: mae, rmse, wape, correlation, mase, rmsse.")

Metric primitives defined: mae, rmse, wape, correlation, mase, rmsse.


### 3.2 Baselines + evaluation harness (`recursive_rollout.py`)

In [4]:
def chain_persistence(anchor_value, n_days):
    """Repeat the anchor day's real value for every day of the chain -- the OLD (D-37, no longer
    the standing convention post-D-80) MASE baseline. Kept for reference/diagnostic use only."""
    return np.full(n_days, float(anchor_value))


def doy_climatology(history_series, target_dates, window=7):
    """Historical mean by day-of-year with a +/- `window`-day circular window, computed from
    `history_series` (real y_observed strictly before the anchor). The standing MASE baseline
    (D-80) -- a baseline's conceptual validity matters more than its own error magnitude, and
    holding one value flat for a year (chain_persistence) isn't a real naive forecaster."""
    doy = np.asarray(history_series.index.dayofyear)
    vals = history_series.values.astype(float)
    global_mean = np.nanmean(vals)
    preds = []
    for d in target_dates:
        td = d.dayofyear
        circ = np.abs(((doy - td + 182) % 365) - 182)
        mask = circ <= window
        v = vals[mask]
        preds.append(np.nanmean(v) if mask.any() and np.isfinite(np.nanmean(v)) else global_mean)
    return np.array(preds)


def lead_time_bin(dates, anchor, bins=BINS):
    """Maps each date to its lead-time bin label (matching bin_metrics's bins exactly), or None
    if outside every bin."""
    lead = np.array([(d - anchor).days for d in dates])
    labels = np.full(len(dates), None, dtype=object)
    for lo, hi in bins:
        m = (lead >= lo) & (lead <= hi)
        labels[m] = f"{lo}-{hi}"
    return labels


def bin_metrics(y_true, y_pred, dates, anchor, y_persist=None, bins=BINS):
    """One row per lead-time bin -- the M5-lesson analogue of "don't blend across the hierarchy",
    binning by lead-time-within-the-chain. y_persist (the MASE/RMSSE naive baseline, same length
    as y_true/y_pred) is typically doy_climatology()'s output under the D-80 convention.
    Columns: bin, n, R2, RMSE, MAE, MASE, RMSSE, WAPE, Correlation."""
    lead = np.array([(d - anchor).days for d in dates])
    rows = []
    for lo, hi in bins:
        m = (lead >= lo) & (lead <= hi) & np.isfinite(y_true)
        if m.sum() < 3:
            rows.append(dict(bin=f"{lo}-{hi}", n=int(m.sum()), R2=np.nan, RMSE=np.nan, MAE=np.nan,
                              MASE=np.nan, RMSSE=np.nan, WAPE=np.nan, Correlation=np.nan))
            continue
        yt, yp = y_true[m], y_pred[m]
        r2 = r2_score(yt, yp) if np.var(yt) > 0 else np.nan
        mae_v = mean_absolute_error(yt, yp)
        mase_v = mase(yt, yp, y_persist[m]) if y_persist is not None else np.nan
        rmsse_v = rmsse(yt, yp, y_persist[m]) if y_persist is not None else np.nan
        rmse_v = rmse(yt, yp)
        wape_v = wape(yt, yp)
        corr_v = correlation(yt, yp)
        rows.append(dict(bin=f"{lo}-{hi}", n=int(m.sum()), R2=round(r2, 3) if np.isfinite(r2) else np.nan,
                          RMSE=round(float(rmse_v), 3) if np.isfinite(rmse_v) else np.nan,
                          MAE=round(float(mae_v), 3),
                          MASE=round(float(mase_v), 4) if np.isfinite(mase_v) else np.nan,
                          RMSSE=round(float(rmsse_v), 4) if np.isfinite(rmsse_v) else np.nan,
                          WAPE=round(float(wape_v), 4) if np.isfinite(wape_v) else np.nan,
                          Correlation=round(float(corr_v), 3) if np.isfinite(corr_v) else np.nan))
    return pd.DataFrame(rows)


print("chain_persistence, doy_climatology, lead_time_bin, bin_metrics defined.")

chain_persistence, doy_climatology, lead_time_bin, bin_metrics defined.


### 3.3 Foundation-model forecasters (`recursive_rollout.py`)

Both are one-shot (not iterative rollouts) -- `tabpfn_time_series`/`tabicl`'s own `predict_df`
predicts the entire 365-day horizon in a single forward pass from a context+future-covariates
dataframe. Both deliberately use `y_observed` (not `y_gapfilled`) as context by default --
avoids the gap-filler-optimism circularity risk flagged for every other model's training target
(overridable via `hist_col` in `run_foundation_sweep`, Section 5, per D-72's own finding that
gap-filled context helps).

In [5]:
TABPFN_V2_GENERIC_CHECKPOINT = "tabpfn-v2-regressor.ckpt"   # original TabPFN-TS paper's generic
# tabular v2 regressor checkpoint (D-81) -- no TS-finetuned v2 checkpoint exists; tabpfn_forecast's
# own default (no override) resolves to a TS-finetuned v3 checkpoint instead.


def tabpfn_v2_model_config():
    """tabpfn_model_config dict that forces the OLD generic v2 checkpoint instead of the
    TS-finetuned v3 default. n_estimators/softmax_temperature NOT overridden -- confirmed (D-81)
    that v2 and v3 share identical defaults for both (8, 0.9), so model_path is the only axis
    that needs to change for a fair v2-vs-v3 comparison."""
    from tabpfn.model_loading import prepend_cache_path
    return {"model_path": prepend_cache_path(TABPFN_V2_GENERIC_CHECKPOINT)}


def tabpfn_forecast(hist_target, hist_covariates, future_covariates, mode="local", quantiles=None,
                     tabpfn_model_config=None):
    """hist_target: pandas Series, real history (gaps allowed as NaN). hist_covariates/
    future_covariates: DataFrames indexed by date, same columns in both. mode="local" (default,
    requires TABPFN_TOKEN) or "client" (cloud). tabpfn_model_config: optional dict, e.g.
    tabpfn_v2_model_config() to force v2 (D-81); default None resolves to v3."""
    import tabpfn_time_series as tts

    pipeline_kwargs = dict(
        tabpfn_mode=tts.TabPFNMode.LOCAL if mode == "local" else tts.TabPFNMode.CLIENT
    )
    if tabpfn_model_config is not None:
        pipeline_kwargs["tabpfn_model_config"] = tabpfn_model_config
    pipeline = tts.TabPFNTSPipeline(**pipeline_kwargs)

    context_df = hist_covariates.copy()
    context_df["timestamp"] = context_df.index
    context_df["target"] = hist_target.reindex(context_df.index).values
    context_df = context_df.reset_index(drop=True)

    future_df = future_covariates.copy()
    future_df["timestamp"] = future_df.index
    future_df = future_df.reset_index(drop=True)

    if quantiles is None:
        preds = pipeline.predict_df(context_df, future_df=future_df)
        preds = preds.reset_index()
        return pd.Series(preds["target"].values, index=pd.to_datetime(preds["timestamp"]))

    preds = pipeline.predict_df(context_df, future_df=future_df, quantiles=list(quantiles))
    preds = preds.reset_index()
    idx = pd.to_datetime(preds["timestamp"])
    out = pd.DataFrame(index=idx)
    out["median"] = preds["target"].values
    for q in quantiles:
        out[q] = preds[q].values if q in preds.columns else np.nan
    return out


def tabicl_forecast(hist_target, hist_covariates, future_covariates, quantiles=None):
    """Same context/future-covariate convention as tabpfn_forecast(). Point estimate uses
    TabICLForecaster's MEDIAN (0.5 quantile) column, not its mean-based 'target' column -- fixed
    2026-07-10 (D-66) after confirming the mean column badly overestimates on this heavily
    right-skewed, spike-dominated flux (2-10x too high in a spot check)."""
    from tabicl import TabICLForecaster

    context_df = hist_covariates.copy()
    context_df["timestamp"] = context_df.index
    context_df["target"] = hist_target.reindex(context_df.index).values
    context_df = context_df.reset_index(drop=True)

    future_df = future_covariates.copy()
    future_df["timestamp"] = future_df.index
    future_df = future_df.reset_index(drop=True)

    forecaster = TabICLForecaster()
    kwargs = {"quantiles": list(quantiles)} if quantiles is not None else {}
    preds = forecaster.predict_df(context_df, future_df=future_df, **kwargs)
    preds = preds.reset_index()
    idx = pd.to_datetime(preds["timestamp"])

    if quantiles is None:
        return pd.Series(preds[0.5].values, index=idx)

    out = pd.DataFrame(index=idx)
    out["median"] = preds[0.5].values
    for q in quantiles:
        out[q] = preds[q].values if q in preds.columns else np.nan
    return out


print("tabpfn_v2_model_config, tabpfn_forecast, tabicl_forecast defined.")

tabpfn_v2_model_config, tabpfn_forecast, tabicl_forecast defined.


## 4. Feature-family configs (F-10/D-67)

The 7-config sweep every foundation-model result in this notebook uses.

In [6]:
FAMILIES = {
    "species": ["fx_cattle_dens", "fx_sheep_dens", "fx_lamb_dens"],
    "arable": ["fx_is_arable"],
    "flow": ["fx_flow_mean", "fx_flow_lag7", "fx_flow_lag14", "fx_flow_lag21", "fx_flow_lag28",
             "fx_flow_roll7", "fx_flow_roll14"],
    "mgmt": ["fx_mgmt_fertN_recency", "fx_mgmt_fertN_rate", "fx_mgmt_lime_recency",
             "fx_mgmt_cultiv_recency", "fx_mgmt_cut_recency", "fx_mgmt_manure_recency"],
    "bodyweight": ["fx_total_liveweight_dens"],
}
ALL_NEW = sorted({c for cols in FAMILIES.values() for c in cols})
BASE_FX = [c for c in fx_all if c not in ALL_NEW]

CONFIGS = {"BASE": BASE_FX}
for fam, cols in FAMILIES.items():
    CONFIGS[f"BASE+{fam}"] = BASE_FX + cols
CONFIGS["BASE+ALL"] = BASE_FX + ALL_NEW

print(f"{len(CONFIGS)} feature configs: {list(CONFIGS.keys())}")

7 feature configs: ['BASE', 'BASE+species', 'BASE+arable', 'BASE+flow', 'BASE+mgmt', 'BASE+bodyweight', 'BASE+ALL']


## 5. MASE convention: climatology, not persistence (D-80)

`chain_persistence()` (one anchor-day value held flat for the full 365-day rollout) was the
original MASE denominator (D-37) but was overridden (D-80): a baseline's conceptual validity
matters more than its own error magnitude, and holding one value flat for a year isn't a real
naive forecaster regardless of how low its own error happens to be. `doy_climatology()`
(day-of-year mean, +/-7-day window) is the standing denominator going forward.

**Fills a real gap**: the climatology-baseline artifact this project has been reusing all session
(`results/_today_climatology_baseline.csv`) had no committed generator script (D-80/D-81 both
flagged this). Built properly here, using the fully inlined `doy_climatology`/`lead_time_bin`
from Section 3 -- no `src/` dependency anywhere in the chain.

In [7]:
def build_climatology_baseline(T, towers=TOWERS, anchor_years=ANCHOR_YEARS, n_days=N_DAYS, bins=BINS):
    """Per (tower, anchor_year, bin) climatology baseline MAE -- the MASE denominator (D-80)."""
    rows = []
    for yr in anchor_years:
        anchor = pd.Timestamp(f"{yr}-12-16")
        target_dates = pd.date_range(anchor + pd.Timedelta(days=1), periods=n_days, freq="D")
        for tower in towers:
            dft = T[tower]
            hist_obs = dft.loc[:anchor, "y_observed"]
            y_true = dft["y_observed"].reindex(target_dates).values
            clim = doy_climatology(hist_obs, target_dates)
            bin_labels = lead_time_bin(target_dates, anchor)
            for lo, hi in bins:
                lbl = f"{lo}-{hi}"
                m = bin_labels == lbl
                yt = y_true[m]; cl = np.asarray(clim)[m]
                valid = np.isfinite(yt) & np.isfinite(cl)
                n = int(valid.sum())
                mae_clim = float(np.mean(np.abs(yt[valid] - cl[valid]))) if n > 0 else np.nan
                rows.append({"tower": tower, "anchor_year": yr, "bin": lbl, "n_clim": n,
                             "MAE_climatology": mae_clim})
    return pd.DataFrame(rows)


CLIM = build_climatology_baseline(T)
CLIM.to_csv(RESULTS / "_today_climatology_baseline.csv", index=False)  # keep D-80's reused artifact in sync
print(f"Climatology baseline: {len(CLIM)} rows, {CLIM.MAE_climatology.notna().sum()} valid")
CLIM.groupby("bin")["MAE_climatology"].mean().round(2)

Climatology baseline: 90 rows, 46 valid


bin
1-7         7.53
181-270    54.84
271-365    37.82
31-90      17.90
8-30       32.30
91-180     50.82
Name: MAE_climatology, dtype: float64

In [8]:
def mase_climatology(summary_df, clim=None, target_filter="observed", model_filter=None):
    """MASE_climatology = MAE_model / MAE_climatology, n-weighted mean per (model, config) --
    D-80's exact convention, pure arithmetic on already-saved MAE/n columns (no reruns needed to
    rescale an existing result to a different baseline)."""
    clim = CLIM if clim is None else clim
    df = summary_df.copy()
    if "target" in df.columns:
        df = df[df.target == target_filter]
    if model_filter is not None:
        df = df[df.model.isin(model_filter)]
    df = df.merge(clim, on=["tower", "anchor_year", "bin"], how="left")
    df = df.dropna(subset=["MAE", "MAE_climatology", "n"])
    df = df[(df.n > 0) & (df.MAE_climatology > 0)]
    df["MASE_climatology"] = df["MAE"] / df["MAE_climatology"]
    rows = []
    for (model, cfg), g in df.groupby(["model", "config"]):
        rows.append({"model": model, "config": cfg,
                     "MASE_climatology": np.average(g["MASE_climatology"], weights=g["n"]),
                     "R2": np.average(g["R2"], weights=g["n"]),
                     "n_total": int(g["n"].sum())})
    return pd.DataFrame(rows)


print("mase_climatology() defined.")

mase_climatology() defined.


## 6. Champion reproduction (live) -- `TabPFN+species`

Reruns the standing champion fresh, all 3 towers x 5 anchors, `BASE+species` config only -- cheap
(~15-20s) and doubles as this notebook's own end-to-end smoke test (proves the fully-inlined
harness above is wired correctly before anything else in this notebook is trusted).

In [9]:
def run_foundation_sweep(model_fn, model_label, configs, hist_col="y_observed", towers=TOWERS,
                          anchor_years=ANCHOR_YEARS, n_days=N_DAYS, tabpfn_kwargs=None):
    """Runs one zero-shot foundation-model forecaster (tabpfn_forecast or tabicl_forecast, both
    Section 3.3) across every (config, tower, anchor) combination. hist_col selects observed- vs
    gap-filled-context (D-72/D-80). Returns a long DataFrame matching bin_metrics()'s own schema +
    target/model/config/anchor_year/tower columns, same shape as every b16_foundation_models_v3*.py
    script's output."""
    tabpfn_kwargs = tabpfn_kwargs or {}
    rows = []
    for yr in anchor_years:
        anchor = pd.Timestamp(f"{yr}-12-16")
        target_dates = pd.date_range(anchor + pd.Timedelta(days=1), periods=n_days, freq="D")
        for tower in towers:
            dft = T[tower]
            hist = dft.loc[:anchor]
            hist_target = hist[hist_col]
            y_true = dft["y_observed"].reindex(target_dates).values
            y_gf = dft["y_gapfilled"].reindex(target_dates).values
            anchor_val = dft.loc[anchor, "y_gapfilled"]
            persist = chain_persistence(anchor_val, n_days)

            for cfg_name, fx_cols in configs.items():
                hist_covariates = hist[fx_cols]
                future_covariates = dft.loc[target_dates, fx_cols]
                try:
                    chain = model_fn(hist_target, hist_covariates, future_covariates, **tabpfn_kwargs)
                    yp = chain.reindex(target_dates).values
                    bm = bin_metrics(y_true, yp, target_dates, anchor, y_persist=persist)
                    bm["target"] = "observed"; bm["model"] = model_label
                    bm["config"] = cfg_name; bm["anchor_year"] = yr; bm["tower"] = tower
                    rows.append(bm)
                    bm_gf = bin_metrics(y_gf, yp, target_dates, anchor, y_persist=persist)
                    bm_gf["target"] = "gapfilled"; bm_gf["model"] = model_label
                    bm_gf["config"] = cfg_name; bm_gf["anchor_year"] = yr; bm_gf["tower"] = tower
                    rows.append(bm_gf)
                except Exception as e:
                    print(f"    T{tower} {yr} {cfg_name} {model_label} SKIPPED: {str(e)[:150]}")
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


print("run_foundation_sweep() defined.")

run_foundation_sweep() defined.


In [10]:
t0 = time.time()
champion_raw = run_foundation_sweep(
    lambda ht, hc, fc: tabpfn_forecast(ht, hc, fc, mode="local"),
    "TabPFN", {"BASE+species": CONFIGS["BASE+species"]},
) if TABPFN_OK else pd.DataFrame()
print(f"Champion reproduction: {time.time()-t0:.0f}s, {len(champion_raw)} rows")

champion_headline = mase_climatology(champion_raw)
print("\nChampion (TabPFN+species), climatology-scored:")
champion_headline.round(3)

GPU 0::   0%|          | 0/1 [00:00<?, ?it/s]

GPU 0:: 100%|██████████| 1/1 [00:01<00:00,  1.87s/it]

GPU 0:: 100%|██████████| 1/1 [00:01<00:00,  1.87s/it]

GPU 0::   0%|          | 0/1 [00:00<?, ?it/s]

GPU 0:: 100%|██████████| 1/1 [00:00<00:00,  1.18it/s]

...

GPU 0::   0%|          | 0/1 [00:00<?, ?it/s]

GPU 0:: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s]

GPU 0:: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s]

Champion reproduction: 18s, 180 rows

Champion (TabPFN+species), climatology-scored:


,model,config,MASE_climatology,R2,n_total
0,TabPFN,BASE+species,0.715,-0.038,2124


**Expected**: MASE_climatology ~ 0.715 (D-80/D-81). If this cell's number lands far from that,
something in the environment/harness has drifted -- investigate before trusting anything below.

## 7. Foundation models -- full 7-config sweep (imported, not rerun)

`TabPFN`/`TabICLv2` (observed- and gap-filled-context) and `TabPFN_v2` (D-81's explicit v2-vs-v3
checkpoint ablation) were all run fresh earlier this same session across the full 7-config x
3-tower x 5-anchor sweep, using the equivalent production scripts (`b16_foundation_models_v3*.py`,
`src/`-importing) -- rerunning them again here would be pure duplicate compute for zero new
information. Imported directly from their result CSVs (data, not code -- doesn't affect this
notebook's self-containment).

In [11]:
FOUNDATION_SOURCES = [
    RESULTS / "b16_foundation_models_v3_summary.csv",          # TabPFN, TabICLv2 (observed-ctx)
    RESULTS / "b16_foundation_models_v3_gf_summary.csv",        # TabPFN_gf, TabICLv2_gf (gf-ctx)
    RESULTS / "b16_foundation_models_v3_tabpfnv2_summary.csv",  # TabPFN_v2 (D-81, observed-ctx)
    RESULTS / "b16_foundation_models_v3_tabpfnv2_gf_summary.csv",  # TabPFN_v2_gf (D-81, gf-ctx)
]
foundation_raw = pd.concat([pd.read_csv(p) for p in FOUNDATION_SOURCES if p.exists()], ignore_index=True)
print(f"Imported {len(foundation_raw)} rows from {len(FOUNDATION_SOURCES)} source files, "
      f"models: {sorted(foundation_raw.model.unique())}")

foundation_headline = mase_climatology(foundation_raw)
foundation_headline.sort_values("MASE_climatology").round(3)

Imported 7560 rows from 4 source files, models: ['TabICLv2', 'TabICLv2_gf', 'TabPFN', 'TabPFN_gf', 'TabPFN_v2', 'TabPFN_v2_gf']


,model,config,MASE_climatology,R2,n_total
29,TabPFN_v2,BASE+ALL,0.712,-0.050,2124
17,TabPFN,BASE+bodyweight,0.715,-0.062,2124
20,TabPFN,BASE+species,0.715,-0.038,2124
31,TabPFN_v2,BASE+bodyweight,0.715,-0.064,2124
25,TabPFN_gf,BASE+flow,0.716,-0.006,2124
15,TabPFN,BASE+ALL,0.717,-0.046,2124
32,TabPFN_v2,BASE+flow,0.718,-0.091,2124
19,TabPFN,BASE+mgmt,0.719,-0.064,2124
41,TabPFN_v2_gf,BASE+species,0.719,-0.008,2124
33,TabPFN_v2,BASE+mgmt,0.719,-0.088,2124


## 8. Tree/SARIMAX/DL models (imported, not rerun)

RF/XGB/LightGBM/SARIMAX/both ensembles (`BASE+ALL` only -- the sole config carried to full
rollout for this family after F-10's Stage-1 signal check found no meaningful gain from any
individual family) and TFT/DLinear/LSTM (all 7 configs, both targets) all involve real model
fitting per anchor/tower (SARIMAX statespace fits, gradient-trained DL) -- genuinely expensive to
refit inside a notebook, and already validated. Imported, not rerun, same as Section 7.

In [12]:
tree_raw = pd.read_csv(RESULTS / "b16_recursive_rollout_v3_all_summary.csv")
dl_raw = pd.read_csv(RESULTS / "b16_dl_models_v3_summary.csv")
print(f"Tree/SARIMAX: {len(tree_raw)} rows, models: {sorted(tree_raw.model.unique())}, "
      f"configs: {sorted(tree_raw.config.unique())}")
print(f"DL: {len(dl_raw)} rows, models: {sorted(dl_raw.model.unique())}, "
      f"configs: {sorted(dl_raw.config.unique())}")

tree_headline = mase_climatology(tree_raw, target_filter=None)  # no target col -- always observed
dl_headline = mase_climatology(dl_raw)
pd.concat([tree_headline, dl_headline]).sort_values("MASE_climatology").round(3)

Tree/SARIMAX: 540 rows, models: ['Ensemble_MASEweighted', 'Ensemble_unweighted', 'LightGBM', 'RF', 'SARIMAX', 'XGB'], configs: ['BASE+ALL']
DL: 3780 rows, models: ['DLinear', 'LSTM', 'TFT'], configs: ['BASE', 'BASE+ALL', 'BASE+arable', 'BASE+bodyweight', 'BASE+flow', 'BASE+mgmt', 'BASE+species']


,model,config,MASE_climatology,R2,n_total
5,XGB,BASE+ALL,0.804,-0.228,2124
1,Ensemble_unweighted,BASE+ALL,0.809,-0.194,2124
0,Ensemble_MASEweighted,BASE+ALL,0.809,-0.194,2124
15,TFT,BASE+ALL,0.812,-0.260,2124
16,TFT,BASE+arable,0.826,-0.403,2124
20,TFT,BASE+species,0.840,-0.368,2124
2,LightGBM,BASE+ALL,0.841,-0.266,2124
3,RF,BASE+ALL,0.847,-0.267,2124
17,TFT,BASE+bodyweight,0.860,-0.408,2124
19,TFT,BASE+mgmt,0.872,-0.581,2124


## 9. Master comparison -- every model/config this project has evaluated, climatology-scored

Combines the live champion reproduction (Section 6), the imported foundation-model results
(Section 7), and the imported tree/SARIMAX/DL results (Section 8) into one table. Verified against
`results/b09_b16_climatology_mase_full_table.csv` (D-80's own full recompute) as a correctness
check -- if this cell's numbers don't match that file, something in this notebook's
(fully-inlined) harness has drifted from the established methodology.

In [13]:
MASTER = pd.concat([foundation_headline, tree_headline, dl_headline], ignore_index=True)
MASTER = MASTER.drop_duplicates(subset=["model", "config"]).sort_values("MASE_climatology").reset_index(drop=True)
MASTER.to_csv(RESULTS / "temp_forecasting_pipeline_master_table.csv", index=False)
print(f"{len(MASTER)} model/config combinations")
MASTER.round(3)

69 model/config combinations


,model,config,MASE_climatology,R2,n_total
0,TabPFN_v2,BASE+ALL,0.712,-0.050,2124
1,TabPFN,BASE+bodyweight,0.715,-0.062,2124
2,TabPFN,BASE+species,0.715,-0.038,2124
3,TabPFN_v2,BASE+bodyweight,0.715,-0.064,2124
4,TabPFN_gf,BASE+flow,0.716,-0.006,2124
...,...,...,...,...,...
64,DLinear,BASE+mgmt,1.216,-1.751,2124
65,DLinear,BASE+ALL,1.227,-1.890,2124
66,DLinear,BASE,1.246,-2.975,2124
67,DLinear,BASE+species,1.259,-2.085,2124


In [14]:
# Verification against D-80's own established full recompute
_ref = pd.read_csv(RESULTS / "b09_b16_climatology_mase_full_table.csv")
_check = MASTER.merge(_ref, on=["model", "config"], suffixes=("_here", "_ref"), how="inner")
_check["diff"] = (_check.MASE_climatology_here - _check.MASE_climatology_ref).abs()
print(f"Matched {len(_check)}/{len(_ref)} rows against the established reference table.")
print(f"Max |diff|: {_check['diff'].max():.4f}  (should be ~0 -- pure arithmetic on the same "
      f"underlying MAE/n columns via an independently re-derived harness, so any real mismatch "
      f"means something drifted between this notebook's inlined functions and src/'s originals)")
_check.sort_values("diff", ascending=False)[["model", "config", "MASE_climatology_here",
                                              "MASE_climatology_ref", "diff"]].head(10)

Matched 55/55 rows against the established reference table.
Max |diff|: 0.0000  (should be ~0 -- pure arithmetic on the same underlying MAE/n columns via an independently re-derived harness, so any real mismatch means something drifted between this notebook's inlined functions and src/'s originals)


,model,config,MASE_climatology_here,MASE_climatology_ref,diff
53,DLinear,BASE+species,1.258901,1.258901,2.220446e-16
47,LSTM,BASE+flow,0.982102,0.982102,1.110223e-16
7,TabPFN_gf,BASE+bodyweight,0.720305,0.720305,1.110223e-16
44,LSTM,BASE+arable,0.965931,0.965931,1.110223e-16
41,LSTM,BASE+species,0.952097,0.952097,1.110223e-16
32,TFT,BASE+arable,0.826132,0.826132,1.110223e-16
43,LSTM,BASE+bodyweight,0.964643,0.964643,1.110223e-16
6,TabPFN_gf,BASE+ALL,0.719976,0.719976,0.000000e+00
8,TabPFN_gf,BASE,0.721321,0.721321,0.000000e+00
9,TabPFN_gf,BASE+arable,0.721321,0.721321,0.000000e+00


**Champion check**: `TabPFN`/`BASE+species` should be the best or tied-best row above
(MASE_climatology ~ 0.715, matching D-80's headline exactly).

---

## 10. New experiments

Space for new forecasting experiments going forward -- mirrors the gap-filling notebook's D1-D8
pattern (each new idea gets its own numbered subsection below, built on top of the shared,
fully-inlined harness above: `T`, `CONFIGS`, `CLIM`, `mase_climatology()`,
`run_foundation_sweep()`, `tabpfn_forecast()`, `tabicl_forecast()`). Standing practice for
anything added here:

1. **Smoke test first** (1 tower, 1 anchor, 1-2 configs) before committing to a full 3x5x7 sweep --
   caught real bugs this way multiple times this session (D-80's gap-fill-source ablation, D-81's
   checkpoint override).
2. **Verify the change actually took effect** where it's not obvious from the code alone (e.g.
   confirm predictions genuinely differ before trusting a "no effect" or "improvement" reading).
3. **MASE is climatology-scored** (Section 5) -- don't silently fall back to persistence.
4. **Full 3-tower coverage by default** (CLAUDE.md) -- a single-tower run is a smoke test, not a
   reportable result.
5. **Stay self-contained** -- new experiment code should build on the functions already defined
   in this notebook (Section 3), not reach back out to `src/` imports, keeping this notebook's
   hands-on, standalone character intact.
6. Log real findings (positive or negative) in `DECISIONS.md`, update `BEST_RESULTS.md` §3 if the
   champion changes.

### 10.1 (placeholder -- next experiment goes here)

In [15]:
# next experiment